# 5. Question 1: can race and ethnicity be decoded from the frozen embeddings?

**Probes.** L2-regularised logistic regressions on the standardised embeddings, cross-fitted over the patient-level five folds (notebook 00, section 0.6):

* a **Black-versus-White** probe (binary), fitted on Black and White patients;
* a **four-group** probe (Hispanic, White, Black, Asian; multinomial), evaluated one-versus-rest.

C is chosen from eight values between 0.0001 and 1 by the validation AUROC. Every patient's most recent ECG is scored once, by probes that never saw any ECG of that patient; the out-of-fold scores are pooled, and 95% CIs come from 1000 patient-bootstrap resamples that are **the same for every encoder**, so that differences between encoders are paired.

**Comparators**
* the same architecture with **random weights** (notebook 04);
* a logistic model on **automated ECG measurements, age and sex**, with median imputation and missing indicators. EchoNext releases ventricular rate, atrial rate, PR interval, QRS duration and QTc; the MIMIC-IV machine measurements give ventricular rate, PR interval, QRS duration and QTc (no atrial rate).

**Paper items:** Fig. 1b, Supplementary Table 3 (all columns except "After LEACE", which comes from notebook 08), Table 2 (question 1).

Run time on one RTX 5090: about 15 minutes for EchoNext and 25 minutes for MIMIC-IV.

In [ ]:
%run ./00_settings.ipynb
OUT = results("05_race_probes")
ENCODERS = list(FMS) + list(RANDS)                     # the three pretrained models and their random-weight versions
MEAS = "meas_lr"                                       # logistic model on ECG measurements, age and sex


def fit_and_save(folder, name, X, D, g_row, **kw):
    """Cross-fitted race probes for one encoder (skipped if done before); the out-of-fold scores are row-level and stay in WORK_DIR."""
    f = folder / f"oof_{name}.npz"
    if f.exists():
        return
    t0 = time.time()
    P4, SB, info = probe_folds(X, D, g_row, **kw)
    np.savez_compressed(f, p4=P4.astype(np.float32), sbw=SB.astype(np.float32))
    save_json(info, folder / f"info_{name}.json")
    print(f"{name}: {time.time() - t0:.0f} s; C (Black vs White) {[i['C_bw'] for i in info]}, C (four groups) {[i['C_4'] for i in info]}")


def report(cohort, D, g_lat, folder):
    """Pooled AUROCs with paired bootstrap CIs, and the gains of each model over its comparators."""
    n4 = int((g_lat >= 0).sum())
    boots = boot_idx(n4, SEED_PROBES)                  # the same resamples for every encoder
    bwm = np.isin(g_lat, [B, W])
    rows, bsave = [], {}
    for name in ENCODERS + [MEAS]:
        z = np.load(folder / f"oof_{name}.npz")
        P4, SB = z["p4"].astype(float), z["sbw"].astype(float)
        assert int((np.isfinite(SB) & (g_lat >= 0)).sum()) == n4, name
        rec, bs = evaluate(P4, SB, g_lat, SEED_PROBES, boots=boots)
        fa = [fast_auc(g_lat[(D["fold_of"] == k) & bwm] == B, SB[(D["fold_of"] == k) & bwm]) for k in range(N_FOLDS)]
        info = load_json(folder / f"info_{name}.json")
        role = "pretrained" if name in FMS else "random weights" if name in RANDS else "ECG measurements, age, sex"
        rows.append({"cohort": cohort, "model": name, "role": role, **rec, "bw_fold_min": min(fa), "bw_fold_max": max(fa),
                     "C_bw": ",".join(f"{i['C_bw']:g}" for i in info), "C_4": ",".join(f"{i['C_4']:g}" for i in info)})
        bsave[name] = bs
    res = pd.DataFrame(rows).set_index("model")
    comps = []
    for fm in FMS:
        for other, what in ((fm + "_random", "FM vs random weights"), (MEAS, "FM vs ECG measurements")):
            for metric, key in (("bw", "bw_auroc"), ("macro", "macro_auroc")):
                d = bsave[fm][metric] - bsave[other][metric]
                d = d[np.isfinite(d)]
                comps.append({"cohort": cohort, "comparison": what, "a": fm, "b": other, "metric": metric, "a_auroc": res.loc[fm, key],
                              "b_auroc": res.loc[other, key], "delta": res.loc[fm, key] - res.loc[other, key], "delta_lo": ci95(d)[0], "delta_hi": ci95(d)[1]})
    return res.reset_index(), pd.DataFrame(comps)

## 5.1 EchoNext

In [ ]:
D = load_design("echonext")
g_row = D["g"][D["pi_of_row"]]                         # group of every ECG (-1 = other or unknown: not used by the race probes)
folder = work("echonext", "race_probes")
for name in ENCODERS:
    fit_and_save(folder, name, emb_echonext(name), D, g_row)
A_rows = echonext_rows()
MEAS_E = ["sex", "ventricular_rate", "atrial_rate", "pr_interval", "qrs_duration", "qt_corrected", "age_at_ecg"]
M = pd.DataFrame({c: pd.to_numeric(A_rows[c], errors="coerce") for c in MEAS_E if c != "sex"})
M.insert(0, "sex", (A_rows["sex"].astype(str).str.lower() == "male").astype(float))
fit_and_save(folder, MEAS, M[MEAS_E].to_numpy(float), D, g_row, do_impute=True)
res_e, comp_e = report("EchoNext", D, D["g"], folder)
res_e.to_csv(OUT / "echonext_probe_results.csv", index=False, encoding="utf-8-sig")
comp_e.to_csv(OUT / "echonext_comparisons.csv", index=False, encoding="utf-8-sig")
res_e[["model", "bw_auroc", "bw_lo", "bw_hi", "macro_auroc", "ovr_white", "ovr_black", "ovr_hispanic", "ovr_asian", "n_black", "n_white"]].round(3)

In [ ]:
comp_e[comp_e.metric == "bw"].round(3)

## 5.2 MIMIC-IV

The same analysis in the MIMIC-IV cohort.

In [ ]:
C = load_mimic_cohort()
Dm = load_design("mimic")
g_row = race_index(C.race6)
rows_m = C.row.to_numpy()
folder = work("mimic", "race_probes")
for name in ENCODERS:
    fit_and_save(folder, name, emb_mimic(name)[rows_m], Dm, g_row)
fit_and_save(folder, MEAS, C[["male", "age", "ventricular_rate", "pr_interval", "qrs_duration", "qt_corrected"]].to_numpy(float), Dm, g_row, do_impute=True)
res_m, comp_m = report("MIMIC-IV", Dm, g_row[Dm["lat_idx"]], folder)
res_m.to_csv(OUT / "mimic_probe_results.csv", index=False, encoding="utf-8-sig")
comp_m.to_csv(OUT / "mimic_comparisons.csv", index=False, encoding="utf-8-sig")
res_m[["model", "bw_auroc", "bw_lo", "bw_hi", "macro_auroc", "ovr_white", "ovr_black", "ovr_hispanic", "ovr_asian", "n_black", "n_white"]].round(3)

In [ ]:
comp_m[comp_m.metric == "bw"].round(3)